# 🧠 NYAS BrainTECH — MTM: Left vs Right EEG Classifier

**Your first working machine-learning prototype (Google Colab edition)**

**Our goal:** use existing EEG recordings to predict whether a person was **instructed to imagine moving their left hand or right hand**, then decide whether the prediction is confident enough to send a **virtual** command.

We will use:

- **Dataset:** PhysioNet EEG Motor Movement/Imagery (`runs 4, 8, 12`)
- **EEG processing:** 8–30 Hz band-pass filtering and short signal segments (epochs)
- **Feature extraction:** Common Spatial Patterns (**CSP**)
- **Classifiers:** Logistic Regression (our baseline), **LDA**, and **SVM**
- **Evaluation:** train, validate, and test using **different participants**
- **Research question:** does abstaining on uncertain predictions reduce incorrect commands, and how many useful commands does it reject?

**Important:** This is analysis of *prerecorded* EEG task data. It does **not** read exact thoughts or control a real robotic arm. Predicted probabilities are not automatically reliable confidence guarantees.

**How to use:** Open the notebook in Google Colab. Click **Runtime → Run all**. Start with the default participant list, then expand to more participants once it works.

Official sources: [PhysioNet EEGMMIDB](https://physionet.org/content/eegmmidb/1.0.0/) · [MNE motor imagery/CSP tutorial](https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html) · [BCI Competition IV](https://www.bbci.de/competition/iv/).

## Step 1. Install tools
`mne` handles EEG recordings, `scikit-learn` trains classifiers, and `joblib` saves your trained model. Colab's normal **CPU** runtime is enough for the first experiment.

In [ ]:
!pip -q install "mne>=1.9,<2" "scikit-learn>=1.5,<2" pandas matplotlib joblib

## Step 2. Import Python libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import joblib

from pathlib import Path
from mne.datasets import eegbci
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay
)

mne.set_log_level("ERROR")
print("MNE version:", mne.__version__)
print("Ready to load EEG data!")

## Step 3. Choose participants and EEG task runs

The EEGMMIDB dataset has many participants. We choose **different people** for the three groups:

- **Train:** subjects 1–5 — the model learns patterns here.
- **Validation:** subject 6 — compare LR/LDA/SVM without seeing the final test people.
- **Test:** subjects 7–8 — evaluate the chosen model only after model selection.

Runs **4, 8, and 12** represent *imagined left vs imagined right fist*; in these specific runs, `T1 = LEFT` and `T2 = RIGHT`. (The labels have other meanings in some other runs.)

In [ ]:
TRAIN_SUBJECTS = [1, 2, 3, 4, 5]
VAL_SUBJECTS = [6]
TEST_SUBJECTS = [7, 8]
RUNS = [4, 8, 12]
DATA_FOLDER = "/content/mne_eeg_data"

print("Train:", TRAIN_SUBJECTS)
print("Validate:", VAL_SUBJECTS)
print("Test:", TEST_SUBJECTS)

## Step 4. Download, filter, and divide EEG signals into examples

For each run:

1. Download an EDF recording.
2. Keep EEG channels and apply an 8–30 Hz filter. This includes the motor-imagery-related mu and beta frequency ranges.
3. Read when a participant was instructed to imagine LEFT (`T1`) or RIGHT (`T2`).
4. Take each **1–3 seconds after a cue** as one short EEG example. This avoids the immediate cue response.
5. Store `X` as `[number_of_examples, EEG_channels, time_samples]` and `y` as `0=LEFT, 1=RIGHT`.

**Note:** The model is learning patterns associated with task instructions, not confirming what the person privately imagined.

In [ ]:
def load_one_subject(subject_number):
    """Get features and task labels from one PhysioNet subject."""
    edf_paths = eegbci.load_data(
        subject_number, RUNS, path=DATA_FOLDER,
        update_path=False, verbose="ERROR"
    )

    all_X, all_y = [], []

    for edf_path in edf_paths:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")
        eegbci.standardize(raw)  # consistent channel labels
        raw.pick("eeg")
        raw.filter(8.0, 30.0, method="iir", verbose="ERROR")

        events, annotation_codes = mne.events_from_annotations(
            raw, verbose="ERROR"
        )
        event_id = {
            "LEFT": annotation_codes["T1"],
            "RIGHT": annotation_codes["T2"],
        }
        epochs = mne.Epochs(
            raw, events, event_id=event_id,
            tmin=1.0, tmax=3.0,
            baseline=None, preload=True, verbose="ERROR"
        )
        X_part = epochs.get_data()  # trials × channels × samples
        y_part = np.where(epochs.events[:, -1] == event_id["LEFT"], 0, 1)
        all_X.append(X_part)
        all_y.append(y_part)

    return np.concatenate(all_X), np.concatenate(all_y)


def load_many_subjects(subject_numbers):
    X_parts, y_parts = [], []
    for subject in subject_numbers:
        X, y = load_one_subject(subject)
        print(f"Subject {subject}: {len(y)} examples, {sum(y == 0)} LEFT, {sum(y == 1)} RIGHT")
        X_parts.append(X)
        y_parts.append(y)
    return np.concatenate(X_parts), np.concatenate(y_parts)

X_train, y_train = load_many_subjects(TRAIN_SUBJECTS)
X_val, y_val = load_many_subjects(VAL_SUBJECTS)
X_test, y_test = load_many_subjects(TEST_SUBJECTS)

print("\nShapes:", X_train.shape, X_val.shape, X_test.shape)
print("Class codes: 0 = LEFT, 1 = RIGHT")

## Step 5. Train our baseline classifier and compare it with LDA and SVM

**CSP does not predict LEFT or RIGHT by itself.** It finds EEG signal patterns that differ between the two tasks. A classifier uses those features to predict the task.

We deliberately place CSP **inside** the scikit-learn pipeline. This ensures CSP is learned from the **training participants only**, not from validation or test people (avoids data leakage).

Models to compare:

- **Logistic Regression:** our original simple, interpretable baseline.
- **LDA:** common for classical motor-imagery EEG pipelines.
- **Linear SVM:** another baseline, with probability estimates enabled for this experiment.

We compare **validation balanced accuracy** and select one model without touching the test data.

In [ ]:
def make_pipeline(classifier):
    return Pipeline([
        ("csp", CSP(n_components=4, reg="ledoit_wolf", log=True,
                    norm_trace=False)),
        ("classifier", classifier),
    ])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000)),
    "LDA": make_pipeline(LinearDiscriminantAnalysis()),
    "Linear SVM": make_pipeline(SVC(kernel="linear", probability=True, random_state=42)),
}

validation_results = []
for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    validation_results.append({
        "Model": name,
        "Validation accuracy": accuracy_score(y_val, pred_val),
        "Validation balanced accuracy": balanced_accuracy_score(y_val, pred_val),
    })

results_df = pd.DataFrame(validation_results)
display(results_df.round(3))

best_name = results_df.sort_values(
    "Validation balanced accuracy", ascending=False, kind="stable"
).iloc[0]["Model"]
best_model = models[best_name]
print("Chosen on VALIDATION participants (not test):", best_name)

## Step 6. Evaluate on **unseen participants** (our final test)

A balanced accuracy of about **0.50** is roughly chance level for evenly weighted LEFT/RIGHT classification. Results above that are encouraging, but one test is not enough to prove the model generalizes to everyone. Don't invent or assume an accuracy score before running the experiment.

In [ ]:
y_pred = best_model.predict(X_test)
print("Selected classifier:", best_name)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, target_names=["LEFT", "RIGHT"],
                            zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["LEFT", "RIGHT"],
    cmap="Blues", colorbar=False
)
plt.title("MTM: predictions on unseen people")
plt.show()

## Step 7. Add MTM's confidence-aware decision layer

A conventional classifier **always** chooses LEFT or RIGHT. Our project's extra idea is to choose **NO ACTION** when uncertain.

Below we use a **demonstration threshold of 0.75** chosen **before** inspecting the test predictions. It is **not a proven safe threshold**. A later research iteration should tune and calibrate it on validation data, not on the final test subjects.

Example: `LEFT 0.91 → LEFT`, but `RIGHT 0.54 → NO ACTION`.

In [ ]:
THRESHOLD = 0.75  # Example threshold, not a clinical/safety guarantee

probabilities = best_model.predict_proba(X_test)
model_classes = best_model.classes_  # expected [0, 1]
winner_columns = probabilities.argmax(axis=1)
predicted_codes = model_classes[winner_columns]
confidence = probabilities.max(axis=1)

accepted = confidence >= THRESHOLD
commands = np.where(
    accepted,
    np.where(predicted_codes == 0, "LEFT", "RIGHT"),
    "NO ACTION"
)

baseline_errors = int(np.sum(predicted_codes != y_test))
accepted_errors = int(np.sum((predicted_codes != y_test) & accepted))
coverage = float(np.mean(accepted))
accepted_accuracy = (
    accuracy_score(y_test[accepted], predicted_codes[accepted])
    if np.any(accepted) else np.nan
)

print("Confidence threshold:", THRESHOLD)
print("Predictions accepted:", int(np.sum(accepted)), "of", len(accepted))
print("Command coverage:", f"{coverage:.1%}")
print("Accuracy among accepted predictions:",
      f"{accepted_accuracy:.1%}" if np.isfinite(accepted_accuracy) else "N/A")
print("Wrong commands without abstention:", baseline_errors)
print("Wrong commands with abstention:", accepted_errors)
print("Rejected / no-action predictions:", int(np.sum(~accepted)))

example_rows = pd.DataFrame({
    "Actual": np.where(y_test == 0, "LEFT", "RIGHT"),
    "Predicted": np.where(predicted_codes == 0, "LEFT", "RIGHT"),
    "Confidence": np.round(confidence, 3),
    "MTM decision": commands,
})
display(example_rows.head(20))

## Step 8. Test the coverage–error trade-off across thresholds

This is central to our research. Rejecting more predictions can reduce the **number** of incorrect accepted commands, but also means the virtual arm responds less often.

**Important:** These threshold comparisons on test data are **exploratory**, not the way to pick a final threshold. For publication-quality evaluation, preselect a threshold using validation participants, then report its performance on untouched test participants.

In [ ]:
rows = []
for t in np.arange(0.50, 0.96, 0.05):
    take = confidence >= t
    wrong_accepted = int(np.sum((predicted_codes != y_test) & take))
    rows.append({
        "Threshold": round(float(t), 2),
        "Coverage": np.mean(take),
        "Incorrect accepted commands": wrong_accepted,
        "Accepted accuracy": (
            np.mean(predicted_codes[take] == y_test[take])
            if np.any(take) else np.nan
        ),
    })
threshold_df = pd.DataFrame(rows)
display(threshold_df.round(3))

fig, ax = plt.subplots()
ax.plot(threshold_df["Threshold"], threshold_df["Coverage"], marker="o", label="Coverage")
ax.plot(threshold_df["Threshold"], threshold_df["Accepted accuracy"], marker="s", label="Accepted accuracy")
ax.set(xlabel="Confidence threshold", ylabel="Fraction", ylim=(0, 1.05),
       title="MTM: accuracy–coverage trade-off")
ax.legend()
plt.show()

## Step 9. Save our trained model

This saves the **CSP + classifier** pipeline. To predict on new EEG recordings later, we must apply **the exact same EEG channel order, filter, and epoching rules** as above. A saved pipeline alone does not preprocess raw files.

Only load saved `.joblib` files you trust; model serialization should be handled like executable code.

In [ ]:
OUTPUT_FILE = "/content/MTM_CSP_EEG_classifier.joblib"
joblib.dump({
    "model": best_model,
    "label_meanings": {0: "LEFT", 1: "RIGHT"},
    "train_subjects": TRAIN_SUBJECTS,
    "validation_subjects": VAL_SUBJECTS,
    "test_subjects": TEST_SUBJECTS,
    "run_numbers": RUNS,
    "filter_hz": (8.0, 30.0),
    "epoch_seconds_after_cue": (1.0, 3.0),
    "demo_confidence_threshold": THRESHOLD,
}, OUTPUT_FILE)
print("Saved model at:", OUTPUT_FILE)

# Uncomment these lines in Google Colab to download the trained model:
# from google.colab import files
# files.download(OUTPUT_FILE)

## How to present this to Akshara and the team

> “We trained a CSP-based EEG classifier on the PhysioNet motor-imagery recordings. We compared Logistic Regression, LDA, and SVM using data from a separate validation participant. Then we evaluated the selected classifier on previously unseen participants. Finally, we introduced a confidence threshold that outputs NO ACTION when the classifier is uncertain. We measured accuracy, incorrect accepted commands, and coverage.”

**Fill in actual results from the cells.** Do not claim results you haven't observed.

### How Akshara's BCI Competition IV dataset fits in

The link shared by Akshara includes multiple datasets, including **2a (22 EEG channels; 4 motor imagery classes)** and **2b (3 EEG channels; left vs right)**. Those recordings are provided separately in GDF format; they are **not automatically mixed** with PhysioNet's 64-channel EDF recordings. To test on BCI IV, we need a separate loader, label mapping, matching channel selection and preprocessing, and fair train/test partitioning. We should first establish a working PhysioNet baseline, then implement the BCI IV pipeline once the team confirms **2a or 2b**.

### Next experiments for a stronger research project

1. Train/test on more participants and repeat with different held-out subjects.
2. Compare logistic regression, LDA, and SVM across multiple subject splits—not just one.
3. Calibrate probabilities and **select the threshold on validation subjects**, then report one final test result.
4. Compare the raw-command system with MTM's NO ACTION mode at the same evaluation data size.
5. Add a virtual-arm simulation **only after** the classification results are clear.

**References:**
- PhysioNet EEG Motor Movement/Imagery: https://physionet.org/content/eegmmidb/1.0.0/
- MNE CSP decoding example: https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html
- BCI Competition IV: https://www.bbci.de/competition/iv/